# RAG Ingestion Layer
### Ein geführter Workshop zur ersten und kritischsten Stufe jedes RAG-Systems

## Bevor wir beginnen: Das große Bild

Bevor eine einzige Zeile Code läuft, lohnt es sich, das System einmal gedanklich
zusammenzusetzen. Wer die Ingestion Layer versteht, versteht auch, warum die späteren
Stufen so aufgebaut sind, wie sie aufgebaut sind.

### Was ist ein RAG-System?

RAG steht für **Retrieval-Augmented Generation**. Die Grundidee:

> Statt ein Sprachmodell mit allem Wissen zu trainieren, das es jemals brauchen könnte,
> lassen wir es zur Laufzeit in einer Wissensbasis **suchen** und geben ihm die
> gefundenen Textstellen als Kontext mit.

Ein RAG-System hat zwei Hälften:

**Offline (einmalig, oder wenn sich die Daten ändern).** Hier werden Rohdaten aufbereitet,
in Vektoren umgewandelt und in einem Index gespeichert. Die Reihenfolge:

1. Rohdaten einlesen
2. Cleaning (deterministische Normalisierung)
3. Chunking (Aufteilen in handhabbare Stücke)
4. Persistenz (Chunks auf Disk schreiben)
5. Embedding (Chunks in Vektoren übersetzen)
6. Indexing (Vektoren durchsuchbar machen)

**Online (bei jeder Nutzerfrage).** Hier wird die eigentliche Anfrage beantwortet:

1. Query einbetten
2. im Vektorindex suchen
3. die besten Chunks zurückgeben
4. das LLM antwortet mit diesen Chunks als Kontext

Dieses Notebook behandelt ausschließlich die ersten vier Schritte der Offline-Hälfte,
also bis einschließlich der Persistenz.

### Warum die Ingestion Layer kritisch ist

Die Ingestion Layer ist die einzige Stelle im System, an der Rohdaten überhaupt
angefasst werden. Alles danach (Embedding, Retrieval, LLM-Antwort) baut auf dem auf,
was hier entsteht. Das klingt nach reiner Vorverarbeitung, ist aber mehr als das.

**Retrieval-Qualität entscheidet sich hier**, nicht im Retrieval-Modul und nicht im LLM.
Die Kette dahinter:

- Ein Embedding-Modell übersetzt einen Textabschnitt in einen Vektor, der seine
  semantische Bedeutung repräsentiert.
- Hat der Text kein klares semantisches Zentrum, weil er mitten im Gedanken anfängt
  oder endet, zu groß oder zu klein ist, wird auch der Vektor unscharf.
- Ein unscharfer Vektor liefert bei der Suche schlechte Treffer.
- Das LLM bekommt dann irrelevanten oder zerstückelten Kontext, und die Antwort wird
  entsprechend schwach.

Wichtig für die Praxis: **Schlechte Chunks lassen sich nachgelagert nicht reparieren.**
Kein Retrieval-Algorithmus, kein Reranker und kein noch so guter Prompt rettet einen
Chunk, der semantisch inkohärent ist. Genau deshalb lohnt es sich, hier zu experimentieren.

### Was passiert in diesem Notebook?

Wir durchlaufen die Ingestion-Pipeline Stück für Stück: Rohdaten, Cleaning, Chunking,
Persistenz, bis hin zu Chunks, die für das Embedding bereit sind. Jeder Schritt ist
einzeln ausführbar und so gebaut, dass sich Parameter gefahrlos verändern lassen.

Was wir hier **nicht** tun: Embedding, Vektordatenbanken, Retrieval, LLM-Inferenz.
Diese Stufen werden nur so weit erwähnt, dass klar bleibt, wo die Ingestion endet.


## Welche Bausteine dieses Notebook verwendet

Alle Pipeline-Komponenten kommen aus dem Paket `rag.ingestion`. Das Notebook importiert
sie nicht alle auf einmal, sondern dort, wo sie gebraucht werden. Zur Orientierung hier
vorab der Überblick, welche Module welche Rolle in der Ingestion Layer spielen:

**Einlesen (Loader).**
`rag.ingestion.loaders.md_loader.MdLoader` und `...jsonl_loader.JsonlLoader` lesen
Markdown- bzw. JSONL-Dateien ein und bringen sie in ein einheitliches Dokumentformat.
Welcher Loader für welche Dateiendung zuständig ist, wird in der Pipeline explizit gebunden.

**Normalisieren (Cleaning).**
`rag.ingestion.cleaner.DefaultCleaner` vereinheitlicht die Form des Textes
(Zeilenenden, BOM, Trailing Whitespace), ohne den Inhalt zu verändern.

**Aufteilen (Chunking).**
`rag.ingestion.chunking.sliding_window_chunker.SlidingWindowChunker` schneidet nach
fester Zeichenzahl, `...document_structure_chunker.DocumentStructureChunker` nutzt die
Dokumentstruktur. Die Konstanten `FIXED_OVERLAP` und `STRUCTURE_AWARE` aus
`...chunking.strategies` markieren, welche Strategie ein Chunk durchlaufen hat.

**Zusammensetzen und Steuern.**
`rag.ingestion.composition` (`IngestionComponents`, `LoaderBinding`, `IngestionRequest`)
beschreibt, aus welchen Teilen die Pipeline besteht.
`rag.ingestion.ingestion_api` (`create_ingestion_service`, `run_ingestion`) führt sie aus.

**Persistenz, Deduplication, Integrität, Metriken.**
`rag.ingestion.storage.jsonl_store.JSONLStore` schreibt Chunks als JSONL,
`...storage.dedup.InMemoryDedupStrategy` verhindert Duplikate,
`...storage.chunk_loader.load_chunks` liest zurück,
`...storage.integrity` prüft Anzahl und IDs, und
`rag.ingestion.metrics.IngestionMetrics` sammelt Kennzahlen über den Lauf.

Dazu kommt etwas Standardbibliothek (`pathlib`, `collections.Counter`, `json`, `hashlib`)
für Dateizugriff, Zählung und Hashing.

Der praktische Wert dieser Aufteilung: Jede Komponente ist austauschbar. Man kann einen
eigenen Loader registrieren, eine andere Chunking-Strategie einhängen oder einen anderen
Store verwenden, ohne den Rest der Pipeline anzufassen.


---

## 1. Rohdaten: Was liegt vor?

Der erste Schritt ist immer derselbe: verstehen, womit wir es zu tun haben.

Ein häufiger Fehler ist, Ingestion-Code zu schreiben, ohne die Eingabe wirklich zu kennen.
Wie viele Dateien? Welche Formate? Wie groß? Wie strukturiert?

Diese Fragen sind keine Formalität. Sie bestimmen, welche Loader und Chunker sinnvoll sind und an welchen Stellen Fallbacks gebraucht werden.

In [ ]:
from pathlib import Path
from collections import Counter

raw_path = Path("/home/jovyan/shared/data/raw")

supported_extensions = {".md", ".jsonl"}
all_files = sorted(raw_path.rglob("*")) if raw_path.exists() else []
files = [f for f in all_files if f.is_file() and f.suffix in supported_extensions]

print(f"Verzeichnis  : {raw_path}")
print(f"Dateien      : {len(files)}")
print()

for f in files:
    print(f"  {f.suffix}  {f.name:<40}  {f.stat().st_size:>8} Bytes")

print()
for ext, count in Counter(f.suffix for f in files).most_common():
    print(f"  {ext}  →  {count} Datei(en)")

**Was dieser Output bedeutet:**

Wir sehen die Dateien, die als Rohdaten vorliegen. Jede Datei wird durch einen
entsprechenden **Loader** eingelesen, etwa der `.md`-Loader für Markdown
und der `.jsonl`-Loader für line-delimited JSON.

Loader tun mehr als `file.read_text()`. Sie normalisieren die Dateistruktur in ein
einheitliches Dokument-Format, das die Pipeline erwartet. Ein `.jsonl`-Loader zum
Beispiel iteriert über Zeilen, parst jede als JSON-Objekt und extrahiert das relevante
Textfeld, damit der Rest der Pipeline nicht wissen muss, wie die Rohdaten aussahen.

In [ ]:
md_files    = [f for f in files if f.suffix == ".md"]
jsonl_files = [f for f in files if f.suffix == ".jsonl"]

sample_md    = md_files[0]    if md_files    else None
sample_jsonl = jsonl_files[0] if jsonl_files else None

print(f"Markdown-Beispiel : {sample_md}")
print(f"JSONL-Beispiel    : {sample_jsonl}")

---

## 2. Rohdokument ansehen: Was ist der Ausgangszustand?

Bevor wir Cleaning und Chunking verstehen, müssen wir sehen, was **vor** diesen
Schritten vorhanden ist.

Rohdaten sind selten so sauber, wie wir es uns wünschen. In der Praxis enthalten sie:

- **Unsichtbare Steuerzeichen** (`\r`, `\r\n`, BOM `\ufeff`), je nach Betriebssystem und Quelle
- **Trailing Whitespace** am Zeilenende, oft aus Copy-Paste oder Editoren
- **Inkonsistente Leerzeilen**, manchmal drei Leerzeilen, manchmal keine
- **Fehlende Strukturierung**: ein 50-KB-Markdown-Dokument ist ein einzelnes, unteiltes Objekt

Das Problem: Kein Embedding-Modell der Welt kann mit einem 50-KB-Text sinnvoll umgehen.
Kontextlimits werden gesprengt. Vektoren werden unpräzise. Deshalb müssen Dokumente
geteilt werden. Darauf gehen wir beim Chunking ein.

In [ ]:
if sample_md is None:
    print("Keine .md Datei gefunden.")
else:
    raw_md = sample_md.read_text(encoding="utf-8")
    print(f"Datei   : {sample_md.name}")
    print(f"Groesse : {len(raw_md)} Zeichen")
    print()
    print("--- Erste 600 Zeichen (als repr, zeigt unsichtbare Zeichen) ---")
    print(repr(raw_md[:600]))

**Was wir im `repr`-Output suchen:**

- `\r\n` statt nur `\n`: Windows-Zeilenenden. Wenn nicht normalisiert, entstehen
  beim späteren Splitting inkonsistente Ergebnisse.
- `\ufeff` am Anfang: UTF-8 BOM. Viele Parser stolpern darüber oder behandeln es als Inhalt.
- `   ` (Leerzeichen am Zeilenende): Schleichen sich in Chunks und beeinflussen Zeichenzahlen.

Diese Zeichen sind mit dem bloßen Auge nicht sichtbar. `repr()` macht sie sichtbar,
deshalb verwenden wir es hier.

In [ ]:
import json

if sample_jsonl is None:
    print("Keine .jsonl Datei gefunden.")
else:
    with open(sample_jsonl, "r", encoding="utf-8") as f:
        first_line = next((l for l in f if l.strip()), None)

    print(f"Datei : {sample_jsonl.name}")
    if first_line:
        try:
            obj = json.loads(first_line)
            print(f"Felder : {list(obj.keys())}")
        except Exception:
            pass
        print(f"Raw    : {repr(first_line[:200])}")

**JSONL ist kein Dokument, sondern ein Stream.**

Eine `.jsonl`-Datei enthält pro Zeile ein eigenständiges JSON-Objekt. Das bedeutet:
Eine einzige `.jsonl`-Datei kann hunderte logisch getrennte Dokumente enthalten.
Der `JsonlLoader` iteriert über diese Zeilen und behandelt jede als eigenständiges Dokument.

Das erklärt auch, warum die Anzahl der Chunks später deutlich höher sein kann als
die Anzahl der Dateien: mehrere Dateien, viele Dokumente, viele Chunks.

---

## 3. Cleaning: Deterministisch normalisieren

### Das Problem, das Cleaning löst

Stellen wir uns vor, wir verarbeiten dasselbe Dokument zweimal: einmal auf einem
Linux-System, einmal auf Windows. Auf Windows hat das Dokument `\r\n` als Zeilenenden.
Wenn wir daraus einen SHA-256-Hash berechnen, erhalten wir **unterschiedliche Hashes**,
obwohl der Inhalt identisch ist.

Das ist das Problem. Eine `document_id` auf Basis eines Hashes ist nur dann zuverlässig,
wenn der Text, aus dem der Hash berechnet wird, **deterministisch** ist,
also immer identisch, egal wann und wo.

### Was Cleaning tut und was es nicht tut

Der `DefaultCleaner` **verändert den Inhalt nicht**. Er normalisiert nur die **Form**:

1. UTF-8 BOM entfernen (`\ufeff`)
2. Zeilenenden vereinheitlichen (`\r\n`, `\r` → `\n`)
3. Trailing Whitespace pro Zeile entfernen
4. Abschließende Leerzeilen entfernen
5. `None` zurückgeben, wenn nach Cleaning nichts übrig bleibt

Er macht kein Stemming, keine Stopwort-Entfernung, keine semantische Transformation.
Das würde Inhalte verändern, was hier nicht erwünscht ist.

**Reihenfolge ist hier nicht beliebig.** BOM muss als erstes entfernt werden,
da sonst die Zeichenende-Normalisierung auf einem Text arbeitet, der noch ein
führendes Sonderzeichen enthält.

In [ ]:
from rag.ingestion.cleaner import DefaultCleaner

cleaner = DefaultCleaner()

if sample_md is None:
    raise FileNotFoundError("Keine .md Datei fuer Cleaning-Demo gefunden.")

raw = sample_md.read_text(encoding="utf-8")
cleaned = cleaner.clean(raw)

print("VORHER (repr, erste 200 Zeichen):")
print(repr(raw[:200]))
print()
print("NACHHER (repr, erste 200 Zeichen):")
print(repr(cleaned[:200]) if cleaned else "None, Dokument war nach Cleaning leer")

**Was wir im Vorher/Nachher-Vergleich sehen sollten:**

- `\r\n` sollte zu `\n` werden
- Abschließende Leerzeichen am Zeilenende (`  \n`) sollten zu `\n` werden
- `\ufeff` am Anfang sollte verschwinden

Wenn die beiden `repr`-Ausgaben identisch aussehen: Das ist auch ein valides Ergebnis,
es bedeutet, dass das Dokument bereits sauber war. Cleaning ist dann eine Null-Operation,
aber eine *deterministische* Null-Operation.

In [ ]:
if cleaned is None:
    raise ValueError("Cleaner hat None zurueckgegeben, Dokument nach Cleaning leer.")

import hashlib

print(f"Laenge RAW        : {len(raw):>8} Zeichen")
print(f"Laenge CLEANED    : {len(cleaned):>8} Zeichen")
print(f"Entfernt          : {len(raw) - len(cleaned):>8} Zeichen")
print()

# Die document_id wird im System exakt so abgeleitet:
doc_id = "doc_" + hashlib.sha256(cleaned.encode("utf-8")).hexdigest()[:16]
print(f"document_id : {doc_id}")
print()
print("Warum das funktioniert:")
print("  Gleicher Input → gleicher SHA-256 → gleiche ID → stabiler Vektorindex.")
print("  Ohne Determinismus: gleiche Datei, zwei Laeufe, zwei IDs → Duplikate im Index.")

### Warum Determinismus systemkritisch ist

Angenommen, Cleaning wäre nicht deterministisch, würde also je nach internem Zustand mal
so und mal anders normalisieren. Dann würde derselbe Input zu unterschiedlichen Texten
führen, und damit zu unterschiedlichen Hashes:

- Erster Lauf: `document_id = "doc_a1b2c3d4..."`
- Zweiter Lauf, gleiche Datei, aber abweichendes Cleaning: `document_id = "doc_x7y8z9w0..."`

Die Folge: Der Vektorindex enthält zwei Einträge für dasselbe Dokument unter verschiedenen
IDs. Deduplication greift nicht mehr, Index-Updates lassen sich nicht mehr gezielt anwenden,
und der Lauf ist nicht reproduzierbar. Deterministisches Cleaning ist deshalb keine
Geschmacksfrage, sondern eine Voraussetzung für ein stabiles System.

**Zum Experimentieren:** Der `DefaultCleaner` ist bewusst minimal. Wer testen will, wie
sich aggressivere Normalisierung auswirkt (etwa Unicode-Normalisierung NFC/NFKC oder das
Zusammenfassen mehrfacher Leerzeilen), kann eine eigene Cleaner-Variante schreiben und sie
an derselben Stelle einhängen. Solange die Operation deterministisch bleibt, bleibt auch
die ID-Logik stabil.


## 4. Chunking: Das schwierigste Problem der Ingestion

### Warum Chunking überhaupt?

Embedding-Modelle können keine beliebig langen Texte verarbeiten. Ein typisches Modell
hat ein Limit von 512 bis 8192 Tokens. Größere Dokumente (ein technisches Handbuch, ein
Gesetzestext, ein Wiki-Artikel) überschreiten das deutlich.

Das Token-Limit ist aber nicht der einzige Grund. Selbst wenn ein Modell ein ganzes
Dokument verarbeiten könnte, wäre der resultierende Vektor **semantisch unscharf**: Ein
Vektor für 20 Seiten Text hat keinen klaren Schwerpunkt, er repräsentiert alles und damit
nichts Bestimmtes. Bei der Suche wird er mit einer präzisen Query verglichen, und das
Match fällt schlecht aus.

### Was ist ein guter Chunk?

Ein guter Chunk erfüllt vier Eigenschaften:

| Eigenschaft | Warum sie wichtig ist |
|-------------|----------------------|
| **Semantisch kohärent** | Der Text behandelt ein Thema. Der Vektor bekommt einen klaren Schwerpunkt. |
| **Vollständig** | Kein abgeschnittener Satz. Das LLM versteht den Chunk, ohne das Original zu sehen. |
| **Weder zu groß noch zu klein** | Zu groß wird unscharf, zu klein verliert Kontext und Signal. |
| **An logischen Grenzen endend** | An Absatz-, Abschnitts- oder Satzgrenzen, dort wo das Dokument selbst eine Grenze setzt. |

### Was ist ein schlechter Chunk?

Nehmen wir diesen Beispieltext:

```
Die Transformerarchitektur basiert auf dem Attention-Mechanismus, der es dem Modell
ermöglicht, relevante Teile des Inputs zu gewichten. Im Gegensatz zu RNNs verarbeitet
ein Transformer alle Token gleichzeitig. Das macht Training deutlich effizienter.

Ein wichtiger Hyperparameter ist die Anzahl der Attention-Heads.
```

Blindes Zeichenschneiden bei Position 150 zerreißt den Sinn:

```
Chunk 1: "Die Transformerarchitektur basiert auf dem Attention-Mechanismus, der es dem"
Chunk 2: "Modell ermöglicht, relevante Teile des Inputs zu gewichten. Im Gegensatz zu"
```

Chunk 1 endet mitten im Satz, Chunk 2 beginnt mit einem Fragment ohne Kontext. Ein
strukturbewusster Schnitt würde den ersten Absatz als Einheit behalten und die einzelne
Folgezeile als eigenen Chunk.

### Warum Overlap existiert

Auch bei guten Grenzen bleibt ein Problem: Der letzte Satz eines Chunks und der erste des
nächsten gehören oft zum selben Gedankenfluss. **Overlap** löst das, indem das Ende eines
Chunks am Anfang des nächsten wiederholt wird. Zwei aufeinanderfolgende Chunks teilen sich
also einen Randbereich.

Der Preis: mehr Chunks, mehr Speicher, mehr Embedding-Aufwand. Der Gewinn: weniger
verlorener Kontext an den Chunk-Grenzen. Wie groß der Overlap sein soll, ist ein
Parameter, mit dem sich gut experimentieren lässt (siehe `overlap` im nächsten Abschnitt).


---

## 5. Chunking-Strategien im Überblick

Es gibt viele Wege, einen Text zu teilen. Jeder hat Stärken, Schwächen und einen Anwendungskontext:

| Strategie | Prinzip | Stärke | Schwäche |
|-----------|---------|--------|----------|
| **Fixed Character Window** | Schneidet bei fixer Zeichenzahl | Simpel, robust | Blind für Satz- und Absatzgrenzen |
| **Sliding Window** | Fixed Window + Overlap | Kein Kontextverlust an Grenzen | Immer noch semantisch blind |
| **Sentence-based** | Schneidet an Satzgrenzen | Sätze bleiben intakt | Erfordert Sentence-Tokenizer, teurer |
| **Structure-aware** | Nutzt Dokumentstruktur (z.B. Markdown-AST) | Semantisch kohärent, natürliche Grenzen | Erfordert parsbares Dokument |
| **Semantic chunking** | Misst semantische Distanz zwischen Sätzen | Präziseste semantische Grenzen | Sehr rechenintensiv, erfordert Embeddings |
| **Token-based** | Schneidet nach Token-Anzahl | Exakt für Modell-Limits | Abhängig vom Tokenizer |

In diesem Notebook verwenden wir **SlidingWindowChunker** und **DocumentStructureChunker**,
einen robusten Fallback und eine strukturbewusste Strategie.

## 6. Strategie 1: SlidingWindowChunker

Der `SlidingWindowChunker` ist die einfachste Strategie: Er schneidet den Text in Fenster
fester Zeichenzahl, mit einem konfigurierbaren Overlap.

**Was er macht:**
- nimmt den Text als flachen String
- erzeugt Chunk 0 von Zeichen 0 bis `chunk_size`
- erzeugt Chunk 1 von `chunk_size - overlap` bis `2 * chunk_size - overlap`
- und so weiter

**Was er nicht weiß:** wo ein Satz endet, wo eine Überschrift steht, wo ein Gedanke
beginnt oder aufhört. Er ist semantisch blind.

**Wann er trotzdem die richtige Wahl ist:** als Fallback für unstrukturierten Text, für
reinen Plain Text ohne Markdown oder HTML, und überall dort, wo kein verlässlicher Parser
verfügbar ist.

**Parameter zum Experimentieren:**
- `chunk_size`: kleiner macht die Vektoren präziser, aber kontextärmer; größer umgekehrt.
- `overlap`: mehr Overlap reduziert Kontextverluste an den Grenzen, erzeugt aber mehr Chunks.
- `strategy`: hier `FIXED_OVERLAP`, der Marker, der später in den Metadaten landet.

Beobachte im Output unten, wo die Chunk-Grenzen liegen. In Produktivsystemen ersetzt man
diese Strategie oft durch satz- oder tokenbasiertes Chunking, sobald die Genauigkeit der
Grenzen wichtiger wird als Einfachheit und Geschwindigkeit.


In [ ]:
from rag.ingestion.chunking.sliding_window_chunker import SlidingWindowChunker
from rag.ingestion.chunking.strategies import FIXED_OVERLAP

if sample_md is None:
    raise FileNotFoundError("Keine .md Datei fuer Chunking-Demo.")

md_text = cleaner.clean(sample_md.read_text(encoding="utf-8"))
if md_text is None:
    raise ValueError("Markdown nach Cleaning leer.")

sliding_chunker = SlidingWindowChunker(
    chunk_size=300,
    overlap=50,
    strategy=FIXED_OVERLAP,
)

md_doc = {
    "id":       "demo_md",
    "content":  md_text,
    "metadata": {"source": str(sample_md), "type": "md"},
}

sliding_chunks = list(sliding_chunker.chunk(md_doc))

print(f"Konfiguration: chunk_size=300, overlap=50")
print(f"Chunks erzeugt: {len(sliding_chunks)}")
print()

for i, chunk in enumerate(sliding_chunks[:3]):
    print(f"--- Chunk {i} ({len(chunk['text'])} Zeichen) ---")
    print(f"Anfang : {repr(chunk['text'][:80])}")
    print(f"Ende   : {repr(chunk['text'][-80:])}")
    print()

**Was im Output zu sehen ist und was es bedeutet:**

Achte auf **Anfang** und **Ende** jedes Chunks:

- Endet Chunk 0 mitten in einem Wort oder Satz? Das ist das Sliding Window ohne
  Bewusstsein für Satzgrenzen.
- Beginnt Chunk 1 mit einem Wortfragment? Das ist die direkte Folge des blinden
  Zeichenschnitts.
- Wie viel Overlap ist zwischen Chunk 0 und Chunk 1 sichtbar?

**Die Abwägung:** Dieser Chunker ist deterministisch, schnell und leicht zu prüfen, aber
seine semantischen Grenzen sind unklar, was direkt die Präzision der späteren Embeddings
beeinflusst.

Wie groß der Schaden in der Praxis ist, hängt vom Text ab: Fließtext ohne viele
strukturelle Grenzen verliert wenig, Dokumente mit klaren Abschnitten, Überschriften und
Listen verlieren deutlich. Genau hier setzt die nächste Strategie an. Wer mag, ändert oben
`chunk_size` und `overlap` und vergleicht, wie sich die Grenzen verschieben.


## 7. Strategie 2: DocumentStructureChunker

Dieser Chunker nutzt die Struktur des Dokuments. Statt blind nach Zeichen zu schneiden,
parst er den Markdown-AST (Abstract Syntax Tree, also die Baumdarstellung der
Dokumentstruktur) und behandelt strukturelle Einheiten als Chunk-Kandidaten:

- Überschriften und ihr Inhalt bleiben zusammen
- Absätze werden als Einheit behandelt
- Listen bleiben intakt
- Codeblöcke werden nicht mitten im Code zerschnitten

**Was passiert, wenn der Text nicht parsbar ist?** Dann greift der **Fallback**: Findet
der Markdown-Parser keine verwertbare Struktur (etwa bei reinem Plain Text), übernimmt
automatisch der `SlidingWindowChunker`. So bekommt man strukturbewusste Chunks, wo möglich,
und robuste Chunks, wo nötig.

**Zum Experimentieren:** Der Fallback-Chunker ist ein Parameter (`fallback_chunker`). Man
kann ihn mit anderem `chunk_size`/`overlap` konfigurieren oder durch eine andere Strategie
ersetzen. Auch `max_chunk_size` lässt sich variieren und bestimmt, wie groß eine
strukturelle Einheit werden darf, bevor sie weiter geteilt wird. In Produktivsystemen
kommen hier oft satz- oder embeddingbasierte (semantische) Chunker zum Einsatz, die
Grenzen noch genauer setzen, dafür aber mehr Rechenzeit kosten.


In [ ]:
from rag.ingestion.chunking.document_structure_chunker import DocumentStructureChunker
from rag.ingestion.chunking.strategies import STRUCTURE_AWARE

structure_chunker = DocumentStructureChunker(
    max_chunk_size=600,
    fallback_chunker=SlidingWindowChunker(
        chunk_size=300,
        overlap=50,
        strategy=FIXED_OVERLAP,
    ),
    strategy=STRUCTURE_AWARE,
)

structure_chunks = list(structure_chunker.chunk(md_doc))

print(f"Konfiguration: max_chunk_size=600, Fallback: SlidingWindowChunker")
print(f"Chunks erzeugt: {len(structure_chunks)}")
print()

for i, chunk in enumerate(structure_chunks[:3]):
    meta     = chunk.get("metadata", {})
    fallback = meta.get("fallback", False)
    strategy = meta.get("strategy", "?")
    print(f"--- Chunk {i} ({len(chunk['text'])} Zeichen | strategy={strategy} | fallback={fallback}) ---")
    print(chunk["text"][:280])
    print()

**Was hier genau zu beobachten ist:**

**1. `fallback=True` in den Metadaten.** Trägt ein Chunk `fallback=True`, dann konnte der
Markdown-Parser diesen Abschnitt nicht sinnvoll strukturieren, und der
`SlidingWindowChunker` hat ihn verarbeitet. Das ist transparent dokumentiert, bedeutet
aber auch: Dieser Chunk kann dieselben Grenzprobleme haben wie ein reiner
Sliding-Window-Chunk.

**2. Chunk-Grenzen direkt prüfen.** Schau dir die erste und letzte Zeile der Chunks an.
Beginnen sie mit einer Überschrift, enden sie mit einem abgeschlossenen Absatz? Das spricht
dafür, dass der Strukturchunker greift.

**3. Grenzen der Strategie.** Auch der `DocumentStructureChunker` liefert nicht in jedem
Fall perfekte Chunks. Ist ein einzelner Absatz länger als `max_chunk_size`, muss er
trotzdem geteilt werden, und das geschieht über den Fallback oder internes Splitting, das
nicht zwingend satzgenau ist. Im Schnitt verbessert die Strategie die Chunk-Qualität
deutlich, eine Garantie für semantische Vollständigkeit ist sie nicht.

Ein interessantes Experiment: Wenn der Fallback-Anteil hier hoch ist, lohnt ein Blick in
die Rohdaten. Vielleicht ist die Markdown-Struktur weniger klar, als man dachte.


### Direkter Vergleich: SlidingWindow vs. DocumentStructure

Beide Strategien laufen über dasselbe Dokument. Wir vergleichen Chunk-Anzahl und
Größenverteilung. So wird sichtbar, ob die strukturbewusste Strategie auf diesen Daten
tatsächlich etwas ändert oder ob der Fallback so oft greift, dass beide Strategien fast
gleich aussehen.


In [ ]:
def chunk_stats(chunks, label):
    lengths = [len(c["text"]) for c in chunks]
    if not lengths:
        print(f"{label}: keine Chunks")
        return
    print(f"--- {label} ---")
    print(f"  Chunks        : {len(lengths)}")
    print(f"  Min (Zeichen) : {min(lengths)}")
    print(f"  Max (Zeichen) : {max(lengths)}")
    print(f"  Avg (Zeichen) : {round(sum(lengths)/len(lengths))}")
    print()

chunk_stats(sliding_chunks,   "SlidingWindowChunker (chunk_size=300, overlap=50)")
chunk_stats(structure_chunks, "DocumentStructureChunker (max_chunk_size=600)")

**Wie wir diese Zahlen interpretieren:**

- **Chunk-Anzahl:** Der `DocumentStructureChunker` erzeugt in der Regel weniger, aber
  größere Chunks, weil er Absätze als Einheit zusammenhält. Wenn die Chunk-Anzahl
  gleich oder ähnlich ist wie beim SlidingWindow, deutet das darauf hin, dass der
  Fallback häufig aktiviert wird. Der Strukturchunker hat dann wenig echte Dokumentstruktur
  vorgefunden.

- **Min/Max-Spanne:** SlidingWindow-Chunks haben eine enge Größenspanne (nahe `chunk_size`).
  StructureChunker-Chunks variieren stärker: kurze Abschnitte bleiben kurz,
  lange bleiben zusammen bis `max_chunk_size`.

- **Wenn Min sehr klein ist:** Ein sehr kleiner Chunk (wenige Zeichen) deutet auf einen
  isolierten Satz oder eine einzelne Überschrift ohne Inhalt hin.
  Solche Chunks sind wenig nützlich für Embedding.

---

## 8. Anatomie eines Chunks

Jeder Chunk ist nicht nur Text. Er trägt Metadaten, die ihn im System eindeutig
identifizierbar und später nachvollziehbar machen.

Schauen wir uns den genauen Aufbau an:

In [ ]:
if structure_chunks:
    c = structure_chunks[0]
    print("Vollständige Chunk-Struktur (erster Chunk):")
    print()
    print(f"  id          : {c['id']}")
    print(f"  document_id : {c['document_id']}")
    print(f"  text[:120]  : {repr(c['text'][:120])}")
    print(f"  metadata    :")
    for key, val in c["metadata"].items():
        print(f"    {key:<25}: {val}")

**Warum jedes Feld existiert:**

| Feld | Wozu es verwendet wird | Was passiert ohne es |
|------|------------------------|----------------------|
| `id` | Primärschlüssel im Vektorindex. Über diese ID wird der Vektor gespeichert und der Chunk-Text später wiedergefunden. | Kein eindeutiger Index-Eintrag möglich. |
| `document_id` | Verbindung zum Quelldokument. Ermöglicht: Alle Chunks eines Dokuments finden, Dokument aktualisieren und nur seine Chunks neu einbetten. | Chunks sind von ihrer Quelle getrennt. |
| `text` | Das, was eingebettet wird. Genau dieser String geht ins Embedding-Modell. | Kein Embedding möglich. |
| `metadata` | Nachvollziehbarkeit: Welches Dokument, welche Strategie, welche Position. | Retrieval-Ergebnisse sind nicht nachvollziehbar. |

**Besonders wichtig:** Die `id` des Chunks wird deterministisch aus `document_id` und
der Position im Dokument abgeleitet. Das bedeutet: Wenn ein Dokument unverändert
neu ingested wird, erhalten die Chunks dieselben IDs, und der Vektorindex kann
gezielt aktualisiert werden, ohne alle anderen Chunks zu berühren.

---

## 9. Pipeline konfigurieren: Alles zusammensetzen

Bisher haben wir die einzelnen Komponenten isoliert betrachtet. Jetzt verbinden wir
sie zu einer Pipeline.

Das Kompositionsmodell dieser Pipeline ist explizit: Jede Komponente wird namentlich
konfiguriert und an ihre Rolle gebunden. Das ist kein Zufall, denn es macht die Pipeline
gezielt diagnostizier- und testbar. Wenn ein Chunk unerwartete Eigenschaften hat, können wir genau
nachvollziehen, welcher Loader, Cleaner oder Chunker ihn erzeugt hat.

**Ein wichtiger Schritt hier:** Wir führen zunächst einen **Trockenlauf** durch
(`persist=False`). Damit verarbeiten wir alle Dokumente vollständig, speichern
aber nichts auf Disk. Das erlaubt uns, die Pipeline zu prüfen, ohne Seiteneffekte zu erzeugen.

In [ ]:
from rag.ingestion.loaders.md_loader import MdLoader
from rag.ingestion.loaders.jsonl_loader import JsonlLoader
from rag.ingestion.composition import IngestionComponents, LoaderBinding, IngestionRequest
from rag.ingestion.ingestion_api import create_ingestion_service, run_ingestion

primary_chunker = DocumentStructureChunker(
    max_chunk_size=600,
    fallback_chunker=SlidingWindowChunker(
        chunk_size=400,
        overlap=80,
        strategy=FIXED_OVERLAP,
    ),
    strategy=STRUCTURE_AWARE,
)

components = IngestionComponents(
    loader_bindings=(
        LoaderBinding(".md",    MdLoader()),
        LoaderBinding(".jsonl", JsonlLoader()),
    ),
    cleaner=cleaner,
    chunker=primary_chunker,
    doc_store=None,
    chunk_store=None,
)

print("Pipeline-Konfiguration:")
print()
for binding in components.loader_bindings:
    print(f"  {binding.extension}  →  {type(binding.loader).__name__}")
print(f"  Cleaner  : {type(components.cleaner).__name__}")
print(f"  Chunker  : {type(components.chunker).__name__}")
print(f"    Fallback: SlidingWindowChunker (chunk_size=400, overlap=80)")

In [ ]:
service     = create_ingestion_service(components)
request     = IngestionRequest(source=raw_path, persist=False)
dry_chunks  = list(run_ingestion(service, request))

print(f"Trockenlauf (persist=False): vollständige Verarbeitung, keine Schreiboperation")
print(f"Chunks gesamt: {len(dry_chunks)}")
print()

from collections import defaultdict
strat_dist = defaultdict(int)
for c in dry_chunks:
    strat_dist[c["metadata"].get("strategy", "?")] += 1

print("Verteilung nach Strategie:")
for strat, count in sorted(strat_dist.items()):
    pct = round(100 * count / len(dry_chunks)) if dry_chunks else 0
    print(f"  {strat:<30} {count:>5} Chunks  ({pct}%)")

**Was uns die Strategie-Verteilung sagt:**

- **Hoher STRUCTURE_AWARE-Anteil:** Die Dokumente haben gut erkennbare Markdown-Struktur.
  Der Chunker konnte sinnvolle Grenzen setzen.
- **Hoher FIXED_OVERLAP-Anteil (Fallback):** Viele Dokumentabschnitte waren nicht
  strukturiert. In solchen Fällen griff der Fallback.

Wenn der Fallback-Anteil überraschend hoch ist: Das ist ein Signal, die Rohdaten zu
untersuchen. Haben die Markdown-Dateien wirklich eine klare Struktur?

**Warum `docs_loaded` > Dateianzahl sein kann:**

Eine `.jsonl`-Datei kann hunderte Zeilen enthalten, und jede Zeile ist ein eigenständiges
Dokument. Der `JsonlLoader` iteriert über alle Zeilen.

## 10. Persistenz: Die Grenze zwischen Ingestion und Embedding

### Warum Persistenz eine eigene Stufe ist

Persistenz ist mehr als Speichern. Sie zieht eine klare Grenze zwischen Ingestion und
Embedding. Ohne diese Grenze müsste das Embedding-Modul bei jedem Lauf erneut Rohdaten
laden, cleanen und chunken. Das hätte drei Nachteile:

1. **Keine unabhängige Evaluierung:** Wären Chunking und Embedding gekoppelt, könnte man
   nicht dasselbe Chunkset mit zwei verschiedenen Embedding-Modellen vergleichen.
2. **Keine Reproduzierbarkeit:** Seiteneffekte im Ingestion-Prozess würden das Ergebnis
   zwischen Läufen schwanken lassen.
3. **Schlechtere Testbarkeit:** Ein Embedding-Modul, das auch chunkt, ist schwerer zu
   testen als eines, das nur liest und einbettet.

Mit Persistenz ist die Aufgabenteilung sauber: Der Ingestion-Lauf erzeugt einmalig eine
stabile `chunks.jsonl`, und der Embedding-Lauf liest genau diese Datei, ohne die Rohdaten
zu kennen. Das Embedding-Modul muss nicht wissen, ob die Quelle Markdown oder JSONL war,
ob ein Fallback aktiv war oder wie die Originaldateien hießen.

### Deduplication

Die `InMemoryDedupStrategy` sorgt dafür, dass dieselbe Chunk-ID nicht zweimal in
`chunks.jsonl` landet, selbst wenn die Pipeline versehentlich zweimal über dasselbe
Dokument läuft. Das verhindert aufgeblähte Stores und doppelte Embeddings. Die Strategie
ist austauschbar: Bei sehr großen Beständen würde man eine persistente Dedup-Variante
einsetzen, die den Zustand nicht nur im Speicher hält.


In [ ]:
from rag.ingestion.storage.jsonl_store import JSONLStore
from rag.ingestion.storage.dedup import InMemoryDedupStrategy
from rag.ingestion.metrics import IngestionMetrics

processed_dir = Path("/home/jovyan/workspace/data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

doc_store_path   = processed_dir / "docs.jsonl"
chunk_store_path = processed_dir / "chunks.jsonl"

# Sauberer Start: Bestehende Dateien entfernen
for p in [doc_store_path, chunk_store_path]:
    if p.exists():
        p.unlink()
        print(f"Vorherige Datei entfernt: {p}")

print(f"Speicherort: {processed_dir}")
print()
print("Zieldateien:")
print(f"  docs.jsonl   : ein Eintrag pro Quelldokument (nach Cleaning)")
print(f"  chunks.jsonl : ein Eintrag pro Chunk (nach Chunking)")
print()
print("Das Embedding-Modul wird später ausschliesslich chunks.jsonl lesen.")

In [ ]:
doc_store   = JSONLStore(doc_store_path,   dedup=InMemoryDedupStrategy())
chunk_store = JSONLStore(chunk_store_path, dedup=InMemoryDedupStrategy())

components_with_store = IngestionComponents(
    loader_bindings=(
        LoaderBinding(".md",    MdLoader()),
        LoaderBinding(".jsonl", JsonlLoader()),
    ),
    cleaner=cleaner,
    chunker=primary_chunker,
    doc_store=doc_store,
    chunk_store=chunk_store,
)

service_with_store = create_ingestion_service(components_with_store)
persist_request    = IngestionRequest(source=raw_path, persist=True)
metrics            = IngestionMetrics()

persisted_chunks = list(run_ingestion(service_with_store, persist_request, metrics=metrics))

print(f"Persistenz-Lauf abgeschlossen.")
print(f"Chunks erzeugt und gespeichert: {len(persisted_chunks)}")
print()
print("Metriken:")
for key, val in metrics.as_dict().items():
    print(f"  {key:<35} {val}")

**Was die Metriken zeigen:**

Die Ingestion-Metriken geben Auskunft über den gesamten Lauf: wie viele Dokumente
geladen, wie viele bereinigt, wie viele verworfen wurden (Cleaning ergab `None`), wie viele
Chunks erzeugt und wie viele dedupliziert wurden.

Wenn `docs_dropped > 0`: Mindestens ein Dokument war nach dem Cleaning leer. Das kann
vorkommen, wenn eine Datei nur Whitespace enthält oder vollständig leer ist.

Wenn `chunks_deduped > 0`: Die Dedup-Strategie hat eingegriffen. In einem sauberen
ersten Lauf sollte das null sein.

---

## 11. Integritätsprüfung: Vertrauen, aber verifizieren

Nach dem Schreiben von Daten auf Disk gibt es keine Garantie, dass alles korrekt
angekommen ist. Dateisystem-Fehler, Encoding-Probleme, abgebrochene Schreiboperationen:
all das kann zu Datenverlust führen, der erst später auffällt.

Deshalb verifizieren wir nach dem Persistenz-Lauf **explizit und hart**: Wir laden
die Datei zurück und vergleichen sie mit dem, was wir geschrieben haben.

**Hard checks statt stillem Ignorieren:** Wenn eine Abweichung gefunden wird, wird ein
`ValueError` geworfen, nicht nur eine Warnung geloggt.

In [ ]:
from rag.ingestion.storage.chunk_loader import load_chunks
from rag.ingestion.storage.integrity import verify_chunk_counts, verify_chunk_ids

reloaded = list(load_chunks(chunk_store_path))

print(f"Chunks geschrieben : {len(persisted_chunks)}")
print(f"Chunks geladen     : {len(reloaded)}")
print()

# verify_chunk_counts: Wirft ValueError wenn Anzahl nicht uebereinstimmt.
verify_chunk_counts(
    expected=len(persisted_chunks),
    path=chunk_store_path,
)

# verify_chunk_ids: Wirft ValueError wenn eine ID fehlt oder eine unerwartete ID auftaucht.
verify_chunk_ids(
    expected_ids={c["id"] for c in persisted_chunks},
    path=chunk_store_path,
)

print("Integritaet OK: Anzahl und IDs stimmen ueberein.")

In [ ]:
# Längenverteilung der Chunk-Texte als Qualitätsindikator
lengths = [len(c["text"]) for c in reloaded]

if lengths:
    print("Laengenverteilung der Chunk-Texte:")
    print()
    print(f"  Anzahl Chunks     : {len(lengths)}")
    print(f"  Min (Zeichen)     : {min(lengths)}")
    print(f"  Max (Zeichen)     : {max(lengths)}")
    print(f"  Avg (Zeichen)     : {round(sum(lengths)/len(lengths))}")
    print(f"  Gesamt (Zeichen)  : {sum(lengths)}")
    print()

    # Sehr kleine Chunks als potenzielles Qualitätsproblem hervorheben
    tiny_chunks = [c for c in reloaded if len(c["text"]) < 50]
    if tiny_chunks:
        print(f"  Hinweis: {len(tiny_chunks)} Chunk(s) unter 50 Zeichen, pruefen ob semantisch nuetzlich.")
        for c in tiny_chunks[:3]:
            print(f"    {repr(c['text'])}")

**Wie wir die Längenverteilung lesen:**

- **Durchschnitt nahe max_chunk_size:** Der Chunker füllt Chunks gut aus.
- **Sehr kleine Min-Werte (< 50 Zeichen):** Einzelne isolierte Elemente, z.B. eine
  Überschrift ohne folgenden Inhalt. Diese Chunks haben wenig semantischen Gehalt
  und produzieren schwache Embeddings.
- **Sehr große Max-Werte (weit über max_chunk_size):** Sollte eigentlich nicht vorkommen.
  Falls doch, deutet es auf ein Konfigurationsproblem hin.

**Zeichenanzahl ≠ semantische Qualität.** Ein 400-Zeichen-Chunk kann brillant oder
wertlos sein, abhängig davon, ob er einen kohärenten Gedanken enthält oder vier
zufällige Satzfragmente.

## 12. Embedding-Bereitschaft: Übergabe an die nächste Stufe

Die Ingestion ist abgeschlossen. Bleibt die formale Übergabe: Wir prüfen, ob `chunks.jsonl`
alle Voraussetzungen erfüllt, die das Embedding-Modul erwartet.

Das Embedding-Modul tut genau eine Sache: Es nimmt `chunk["text"]`, schickt ihn durch das
Embedding-Modell und erhält einen Vektor, der später in den Vektorindex wandert. Dafür
braucht jeder Chunk:

- `text`: nicht leer, ein String
- `id`: vorhanden und eindeutig
- `document_id`: für die spätere Verknüpfung
- `metadata`: mit `strategy`-Feld für die Nachvollziehbarkeit

Was es **nicht** braucht: Rohdaten, Loader, Cleaner, Chunker. Es kennt nicht einmal die
Namen der ursprünglichen Dateien. Genau diese Entkopplung macht es möglich, später das
Embedding-Modell zu wechseln, ohne die Ingestion erneut auszuführen.


In [ ]:
n   = len(reloaded)
ids = [c["id"] for c in reloaded]

checks = {
    "chunks.jsonl vorhanden"        : chunk_store_path.exists(),
    "Mindestens 1 Chunk"            : n > 0,
    "text nicht-leer"               : all(isinstance(c["text"], str) and c["text"] for c in reloaded),
    "id vorhanden"                  : all(c["id"] for c in reloaded),
    "IDs eindeutig"                 : len(set(ids)) == n,
    "document_id vorhanden"         : all(c["document_id"] for c in reloaded),
    "metadata vorhanden"            : all(isinstance(c["metadata"], dict) for c in reloaded),
    "metadata[strategy] vorhanden"  : all("strategy" in c["metadata"] for c in reloaded),
}

all_ok = all(checks.values())

print("Embedding-Readiness-Check:")
print()
for label, result in checks.items():
    status = "OK  " if result else "FAIL"
    print(f"  [{status}]  {label}")

print()
if not all_ok:
    raise RuntimeError("Embedding-Readiness-Check fehlgeschlagen, siehe FAIL oben.")

print(f"EMBEDDING-READY  ({n} Chunks in {chunk_store_path})")

## 13. Zusammenfassung: Was wir gelernt haben

### Cleaning: Determinismus als Grundlage

Cleaning ist keine kosmetische Vorverarbeitung, sondern die Basis für reproduzierbare IDs,
stabile Indizes und zuverlässige Deduplication. Gleicher Input ergibt denselben SHA-256 und
damit dieselbe `document_id`. Ohne Determinismus bricht das gesamte Update- und
Dedup-Modell.

### Chunking: Wo Retrieval-Qualität entschieden wird

Ein Embedding-Modell ist nur so gut wie der Text, den es bekommt. Ein semantisch
inkohärenter Chunk liefert einen unscharfen Vektor, und ein unscharfer Vektor führt zu
schlechten Treffern, die kein Reranker und kein Prompt im Nachhinein repariert. Der
`DocumentStructureChunker` verbessert die Qualität spürbar, ist aber kein Allheilmittel.
Chunks aus dem Fallback tragen `strategy=FIXED_OVERLAP` in den Metadaten und sind dadurch
jederzeit erkennbar.

### Persistenz: Die klare Grenze

Ingestion erzeugt `chunks.jsonl`, Embedding liest `chunks.jsonl`. Diese Grenze erlaubt drei
Dinge: verschiedene Embedding-Modelle auf denselben Chunks zu evaluieren, Ingestion und
Embedding getrennt zu testen, und Läufe reproduzierbar zu halten.

### Was du anpassen kannst

Der Baukasten ist an mehreren Stellen austauschbar:

- **Loader:** für weitere Dateiformate (HTML, PDF-Text, CSV) eigene Loader registrieren.
- **Cleaner:** stärkere oder schwächere Normalisierung, solange sie deterministisch bleibt.
- **Chunker:** `chunk_size`, `overlap`, `max_chunk_size` variieren oder eine ganz andere
  Strategie einhängen (satzbasiert, tokenbasiert, semantisch).
- **Store und Dedup:** für große Bestände auf persistente Varianten umstellen.

### Was als nächstes kommt

Die nächste Stufe liest `chunks.jsonl` und arbeitet so weiter:

1. `load_chunks(chunk_store_path)` lädt die Chunks.
2. Das Embedding-Modell übersetzt `chunk["text"]` in einen Vektor.
3. Der Vektorindex speichert den Vektor unter `chunk["id"]` zusammen mit den Metadaten.
4. Retrieval bettet eine Query ein, sucht im Index und gibt die besten Chunks zurück.
5. Das LLM bekommt die Chunk-Texte als Kontext und formuliert die Antwort.

Damit ist die Ingestion abgeschlossen. Alles Folgende baut auf der Qualität auf, die hier
festgelegt wurde.
